# Walk-forward daily limit validation

Select a discount using only a trailing training window, then score that fixed discount on the following unseen year.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from retail_sp500.daily_data import daily_data_summary, load_or_fetch_twelve_data_daily

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

from retail_sp500.limit_portfolio import walk_forward_recurring_limit_selection

In [ ]:
SYMBOL = "SPY"
START_DATE = "2007-06-01"
CACHE_PATH = Path("data/processed/spy_daily_1day.csv")
REFRESH = False

daily = load_or_fetch_twelve_data_daily(
    os.getenv("TWELVE_DATA_API_KEY"),
    cache_path=CACHE_PATH,
    refresh=REFRESH,
    symbol=SYMBOL,
    start_date=START_DATE,
)

source = daily_data_summary(daily, symbol=SYMBOL)
print(source)
assert source["interval"] == "1day"
assert daily.index.max() <= pd.Timestamp.today().normalize()
daily.tail()

## Walk-forward configuration

In [ ]:
DISCOUNTS = np.arange(0.0, 0.0301, 0.001)
TRAIN_YEARS = 5
TEST_YEARS = 1
MAX_WAIT_SESSIONS = 5

walk_forward = walk_forward_recurring_limit_selection(
    daily,
    DISCOUNTS,
    train_years=TRAIN_YEARS,
    test_years=TEST_YEARS,
    step_years=1,
    max_wait_sessions=MAX_WAIT_SESSIONS,
    monthly_contribution=1_000,
)

walk_forward

## Out-of-sample summary

In [ ]:
walk_forward_summary = pd.Series({
    "folds": len(walk_forward),
    "median_selected_discount": walk_forward["selected_discount"].median(),
    "mean_test_excess_value": walk_forward["test_ending_excess_value"].mean(),
    "median_test_excess_value": walk_forward["test_ending_excess_value"].median(),
    "positive_test_fold_rate": (walk_forward["test_ending_excess_value"] > 0).mean(),
    "mean_test_forced_fill_rate": walk_forward["test_forced_fill_rate"].mean(),
    "mean_test_wait_sessions": walk_forward["test_average_wait_sessions"].mean(),
})

walk_forward_summary

In [ ]:
px.bar(
    walk_forward,
    x="test_start",
    y="test_ending_excess_value",
    color="selected_discount",
    title="Out-of-sample execution value by test year",
).show()

In [ ]:
px.histogram(
    walk_forward,
    x="selected_discount",
    nbins=len(DISCOUNTS),
    title="Discounts selected by trailing five-year windows",
).show()

## Decision rule

Prefer a stable region that remains positive across many unseen folds. Do not select a single sharp in-sample maximum.